# (01) Import and settings

In this section we import the required libraries, configure display options and fix the random seed for reproducibility.


In [ ]:
# (01) Import and settings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Plot style
sns.set(style="whitegrid")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


: 

# (02) Loading dataset

We load `titanic.csv` from the local folder, inspect its structure and explicitly define `Survived` as the binary target column. All other columns are treated as candidate predictors.


In [ ]:
# (02) Loading dataset

# Load Titanic dataset
data = pd.read_csv("titanic.csv")

print("Shape:", data.shape)
print("\nInfo:")
print(data.info())

print("\nHead:")
display(data.head())

# Target and features
TARGET_COL = "Survived"
assert TARGET_COL in data.columns, "Target column 'Survived' not found."

X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL].astype(int)

print("\nTarget distribution:")
display(y.value_counts(normalize=True).rename("ratio").to_frame())


# (03) Explorative Data Analysis

We take a compact look at the data:

- Descriptive statistics
- Missing value overview, confirming that only `Embarked`, `Age` and `Cabin` contain missing values
- A few key plots to understand survival patterns and age distribution




In [ ]:
# (03) Explorative Data Analysis

# Basic statistics
print("Numeric summary:")
display(X.describe())

print("\nCategorical summary (top categories):")
display(X.describe(include="object"))

# Missing values overview
missing_counts = X.isna().sum().sort_values(ascending=False)
print("\nMissing values per column:")
display(missing_counts.to_frame(name="missing_count"))

# Confirm that only Embarked, Age and Cabin have missing values
cols_with_missing = [col for col in X.columns if X[col].isna().any()]
print("\nColumns with missing values:", cols_with_missing)

expected_missing = {"Embarked", "Age", "Cabin"}
if set(cols_with_missing) == expected_missing:
    print("Confirmation: Only 'Embarked', 'Age' and 'Cabin' contain missing values.")
else:
    print("Note: Unexpected missing pattern compared to the assumption for this lab.")


In [ ]:
# Simple helper for safe plotting
def safe_countplot(x, hue=None, data=None, title=None, figsize=(6, 4)):
    if x in data.columns:
        plt.figure(figsize=figsize)
        sns.countplot(data=data, x=x, hue=hue)
        plt.title(title or f"{x} distribution")
        plt.tight_layout()
        plt.show()

# Combine X and y for plotting
df_plot = X.copy()
df_plot[TARGET_COL] = y

# Survival by Sex
safe_countplot(
    x="Sex",
    hue=TARGET_COL,
    data=df_plot,
    title="Survival by sex"
)

# Survival by Pclass
safe_countplot(
    x="Pclass",
    hue=TARGET_COL,
    data=df_plot,
    title="Survival by passenger class"
)

# Age distribution by survival if Age exists
if "Age" in df_plot.columns:
    plt.figure(figsize=(6, 4))
    sns.kdeplot(
        data=df_plot[df_plot["Age"].notna()],
        x="Age",
        hue=TARGET_COL,
        common_norm=False,
        fill=True,
        alpha=0.4,
    )
    plt.title("Age distribution by survival")
    plt.tight_layout()
    plt.show()

# Fare distribution
if "Fare" in df_plot.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_plot["Fare"], bins=40)
    plt.title("Fare distribution")
    plt.tight_layout()
    plt.show()


### EDA observations

- Survival is strongly related to `Sex` and `Pclass` (typical pattern: more women and first class passengers survive).
- Younger passengers can have different survival chances compared to older passengers.
- `Fare` has a skewed distribution with a long tail, so robust imputation and scaling are helpful.
- Missing values are limited to `Embarked`, `Age` and `Cabin`, which we handle with explicit domain logic in the next section.


# (04) Feature Engineering (Imputation, missing values, new features, encoding, scaling, cleanup)

Goal: build a clean, reusable preprocessing pipeline that:

- Implements custom domain logic for `Embarked`, `Age` and `Cabin`
- Uses title information extracted from `Name`
- Creates compact and meaningful numeric and categorical features
- Produces a model ready matrix that works well for both KNN and DecisionTreeClassifier


## 4.1 Missing value handling

**Embarked**

For missing `Embarked` values:

1. Try to impute from other passengers with the same `Ticket` and known `Embarked` using the mode of that ticket group.
2. If no such information exists, use a heuristic based on `Pclass` and `Fare`:
   - First class with very high fare (for example `Fare >= 70`) → `C`
   - First class with medium high fare (for example `Fare >= 30`) → `S`
   - Third class with low fare (for example `Fare <= 15`) → `Q`
   - Otherwise → `S` as a reasonable default

**Age**

1. Extract a `Title` from `Name` using a regular expression (for example `Mr`, `Mrs`, `Miss`, `Master`, `Dr`).
2. Compute the global median age from all passengers with known `Age`.
3. Compute the median age per `Title` from rows with known `Age`.
4. Impute missing `Age` values row by row:
   - If `Age` is known, keep it.
   - If `Age` is missing and the passenger has a `Title` with a known median, use that median.
   - Otherwise, fall back to the global median age.
5. Clip imputed ages to be at least 0.

**Cabin**

- Treat `Cabin` as mostly missing and noisy.
- Create:
  - `CabinDeck` from the first letter of the cabin string
  - `HasCabin` as a binary flag (1 if cabin is present, 0 otherwise)
- Use `"U"` for unknown decks.


## 4.2 Title based feature engineering

From the extracted `Title` we create a model friendly `TitleGroup`:

- `Mr`
- `Mrs` for titles like `Mrs`, `Mme`
- `Miss` for titles like `Miss`, `Ms`, `Mlle`
- `Master`
- `Royalty` for noble titles
- `Officer` for military or professional ranks
- `Rare` for all remaining infrequent titles

Additional simple flags can be derived, for example `IsChild` or `IsMarriedWoman`, as long as they stay interpretable. Title and title group can be useful predictors because they encode social status, gender and age related patterns that correlate with survival.


## 4.3 General feature engineering design

We create at least 12 final features and separate them into numeric and categorical sets.

Examples:

- Numeric: `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`, `FamilySize`, `TicketGroupSize`, `IsAlone`, `IsChild`, `HasCabin`
- Categorical: `Sex`, `Embarked`, `Title`, `TitleGroup`, `CabinDeck`, `TicketPrefix`

The custom transformer will:

- Implement all missing value handling for `Embarked`, `Age` and `Cabin`
- Create titles and grouped titles
- Create family and ticket related features
- Drop unused columns so that the downstream `ColumnTransformer` receives a compact DataFrame


In [ ]:
# (04) Feature Engineering: custom transformer

class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that:
    - Handles missing values for Embarked, Age and Cabin using explicit logic
    - Extracts Title and TitleGroup from Name
    - Creates cabin related features (CabinDeck, HasCabin)
    - Creates family and ticket related features
    - Returns a clean DataFrame with numeric and categorical features
    """

    def __init__(self):
        self.global_age_median_ = None
        self.title_age_median_ = None
        self.title_group_map_ = None
        self.feature_names_in_ = None

    def _extract_raw_title(self, name_series):
        # Extract pattern like "Lastname, Title. Firstname"
        # Example regex: text between comma and dot, use letters only
        return name_series.str.extract(r",\s*([A-Za-z]+)\.", expand=False)

    def _build_title_group_map(self):
        # Mapping of titles to grouped categories
        title_group_map = {
            "Mr": "Mr",
            "Mrs": "Mrs",
            "Mme": "Mrs",
            "Miss": "Miss",
            "Ms": "Miss",
            "Mlle": "Miss",
            "Master": "Master",
            "Lady": "Royalty",
            "Countess": "Royalty",
            "Sir": "Royalty",
            "Don": "Royalty",
            "Dona": "Royalty",
            "Jonkheer": "Royalty",
            "Col": "Officer",
            "Major": "Officer",
            "Capt": "Officer",
            "Dr": "Officer",
            "Rev": "Officer",
        }
        return title_group_map

    def fit(self, X, y=None):
        # Ensure DataFrame
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = list(X.columns)

        # Title and age medians
        if "Name" in X.columns and "Age" in X.columns:
            name_series = X["Name"].astype(str)
            age_series = X["Age"]

            # Extract raw title
            raw_titles = self._extract_raw_title(name_series)

            # Global age median
            self.global_age_median_ = age_series[age_series.notna()].median()

            # Median age per title, using rows with known Age
            known_age_mask = age_series.notna()
            title_for_age = raw_titles[known_age_mask]
            age_for_age = age_series[known_age_mask]

            self.title_age_median_ = (
                age_for_age.groupby(title_for_age)
                .median()
                .dropna()
            )
        else:
            # Fallback in case columns are missing
            self.global_age_median_ = 30.0
            self.title_age_median_ = pd.Series(dtype=float)

        # Title group mapping
        self.title_group_map_ = self._build_title_group_map()

        return self

    def _impute_embarked(self, df):
        df = df.copy()
        if "Embarked" not in df.columns:
            return df

        embarked = df["Embarked"]
        missing_mask = embarked.isna()

        if not missing_mask.any():
            return df

        # Step 1: mode within each ticket group using known Embarked
        if "Ticket" in df.columns:
            known = df[~df["Embarked"].isna()]
            ticket_modes = (
                known.groupby("Ticket")["Embarked"]
                .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
            )

            # Map ticket level modes to missing rows
            df.loc[missing_mask, "Embarked"] = df.loc[missing_mask, "Ticket"].map(ticket_modes)

            # Update mask in case some remain missing
            missing_mask = df["Embarked"].isna()

        # Step 2: heuristic based on Pclass and Fare
        if missing_mask.any():
            pclass = df.loc[missing_mask, "Pclass"] if "Pclass" in df.columns else 3
            fare = df.loc[missing_mask, "Fare"] if "Fare" in df.columns else 0.0

            # Apply row wise heuristic
            def embarked_heuristic(row):
                p = row.get("Pclass", 3)
                f = row.get("Fare", 0.0)
                if p == 1 and f >= 70:
                    return "C"
                elif p == 1 and f >= 30:
                    return "S"
                elif p == 3 and f <= 15:
                    return "Q"
                else:
                    return "S"

            df.loc[missing_mask, "Embarked"] = df[missing_mask].apply(embarked_heuristic, axis=1)

        return df

    def _impute_age(self, df, raw_title):
        df = df.copy()
        if "Age" not in df.columns:
            return df

        ages = df["Age"].copy()
        missing_mask = ages.isna()

        if not missing_mask.any():
            return df

        # Map each title to its median age if available
        title_medians = raw_title.map(self.title_age_median_)

        # For missing ages, use title median, then global median
        ages_imputed = ages.copy()
        ages_imputed[missing_mask] = (
            title_medians[missing_mask].fillna(self.global_age_median_)
        )

        # Clip ages at sensible lower bound
        ages_imputed = ages_imputed.clip(lower=0)
        df["Age"] = ages_imputed

        return df

    def _handle_cabin(self, df):
        df = df.copy()
        if "Cabin" not in df.columns:
            df["CabinDeck"] = "U"
            df["HasCabin"] = 0
            return df

        # HasCabin flag
        df["HasCabin"] = df["Cabin"].notna().astype(int)

        # CabinDeck from first letter, "U" if unknown
        cabin_str = df["Cabin"].fillna("U").astype(str)
        deck = cabin_str.str[0]
        deck = deck.replace("U", "U")
        df["CabinDeck"] = deck

        return df

    def _create_family_and_ticket_features(self, df):
        df = df.copy()

        # Family size and IsAlone
        if "SibSp" in df.columns and "Parch" in df.columns:
            df["FamilySize"] = df["SibSp"].fillna(0) + df["Parch"].fillna(0) + 1
        else:
            df["FamilySize"] = 1

        df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

        # TicketGroupSize and TicketPrefix
        if "Ticket" in df.columns:
            ticket_counts = df["Ticket"].value_counts()
            df["TicketGroupSize"] = df["Ticket"].map(ticket_counts).fillna(1)

            ticket_clean = (
                df["Ticket"]
                .astype(str)
                .str.replace(r"[./]", " ", regex=True)
                .str.split()
            )

            def prefix_from_parts(parts):
                for p in parts:
                    if not p.isdigit():
                        return p
                return "None"

            df["TicketPrefix"] = ticket_clean.apply(prefix_from_parts)
        else:
            df["TicketGroupSize"] = 1
            df["TicketPrefix"] = "None"

        return df

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        # Ensure key columns exist with reasonable defaults
        for col, default in [
            ("Name", ""),
            ("Embarked", np.nan),
            ("Age", np.nan),
            ("Cabin", np.nan),
            ("Pclass", 3),
            ("Fare", 0.0),
            ("Sex", "Unknown"),
            ("SibSp", 0),
            ("Parch", 0),
        ]:
            if col not in X.columns:
                X[col] = default

        # Extract raw Title
        raw_title = self._extract_raw_title(X["Name"].astype(str))
        raw_title = raw_title.fillna("Rare")
        X["Title"] = raw_title

        # Age imputation using Title and global median
        X = self._impute_age(X, raw_title)

        # Embarked imputation using ticket groups and heuristic
        X = self._impute_embarked(X)

        # Cabin handling (deck and flag)
        X = self._handle_cabin(X)

        # Family and ticket related features
        X = self._create_family_and_ticket_features(X)

        # IsChild as age based flag
        X["IsChild"] = (X["Age"] < 16).astype(int)

        # TitleGroup feature
        def map_title_group(t):
            if t in self.title_group_map_:
                return self.title_group_map_[t]
            else:
                return "Rare"

        X["TitleGroup"] = X["Title"].map(map_title_group)

        # Optional social feature: IsMarriedWoman (e.g. Mrs)
        X["IsMarriedWoman"] = ((X["TitleGroup"] == "Mrs") & (X["Sex"] == "female")).astype(int)

        # Drop raw text heavy or id like columns
        cols_to_drop = [c for c in ["PassengerId", "Name", "Cabin"] if c in X.columns]
        X = X.drop(columns=cols_to_drop)

        return X


### Define final numeric and categorical feature sets

We now define which columns will be treated as numeric and which as categorical after the custom transformer. Binary flags are kept as numeric so they can be scaled for KNN if needed.


In [ ]:
# Define lists of numeric and categorical features after TitanicFeatureEngineer

numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "TicketGroupSize",
    "IsAlone",
    "IsChild",
    "HasCabin",
    "IsMarriedWoman",
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title",
    "TitleGroup",
    "CabinDeck",
    "TicketPrefix",
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Preprocessing: scale numeric features and one hot encode categorical features
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

print("\nPreprocessor defined.")


# (05) Train test split

We now split the data into training and test sets:

- Training set: used for model training, cross validation and tuning.
- Test set: kept separate and used once at the end for unbiased evaluation.

The split is stratified by `Survived` to preserve the target distribution in both sets.


In [ ]:
# (05) Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
display(y_train.value_counts(normalize=True).rename("ratio").to_frame())

print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).rename("ratio").to_frame())


# (06) ML pipeline (KNN and Decision Tree)

We build two complete pipelines:

1. `KNN` pipeline
2. `DecisionTree` pipeline

Each pipeline:

- Applies `TitanicFeatureEngineer` (custom imputation and feature creation)
- Applies `preprocessor` (scaling numeric features, encoding categorical ones)
- Trains the classifier

This avoids code duplication and keeps preprocessing fully integrated with the models.


In [ ]:
# (06) ML pipeline (KNN and Decision Tree)

def make_pipeline(model):
    return Pipeline(
        steps=[
            ("feature_engineering", TitanicFeatureEngineer()),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

knn_pipeline = make_pipeline(
    KNeighborsClassifier()
)

dt_pipeline = make_pipeline(
    DecisionTreeClassifier(random_state=RANDOM_STATE)
)

print("KNN pipeline:")
print(knn_pipeline)

print("\nDecision Tree pipeline:")
print(dt_pipeline)


# (07) Cross validation and fit of base models

We evaluate both pipelines using stratified k fold cross validation on the training set. We use accuracy as the primary metric and inspect the distribution, mean and standard deviation of scores.

After the cross validation step, we refit each base pipeline on the full training set.


In [ ]:
# (07) Cross validation and fit of base models

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

def evaluate_cv(pipeline, X, y, name="model"):
    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    print(f"{name} CV accuracy scores:", scores)
    print(f"{name} CV mean accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores

print("Base KNN:")
knn_cv_scores = evaluate_cv(knn_pipeline, X_train, y_train, name="KNN")

print("\nBase Decision Tree:")
dt_cv_scores = evaluate_cv(dt_pipeline, X_train, y_train, name="DecisionTree")

# Visual comparison of CV scores
cv_df = pd.DataFrame(
    {
        "KNN": knn_cv_scores,
        "DecisionTree": dt_cv_scores,
    }
)

plt.figure(figsize=(6, 4))
sns.boxplot(data=cv_df)
plt.ylabel("Accuracy")
plt.title("Cross validation accuracy (base models)")
plt.tight_layout()
plt.show()

# Fit base models on full training set
knn_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)


# (08) Hyperparameter tuning

We tune both models using `GridSearchCV` with cross validation on the training set.

Design choices:

- KNN:
  - `n_neighbors`: 3, 5, 7, 9
  - `weights`: uniform, distance
  - `p`: 1 (Manhattan) or 2 (Euclidean)
- DecisionTreeClassifier:
  - `max_depth`: None, 3, 5, 7, 9
  - `min_samples_split`: 2, 5, 10
  - `min_samples_leaf`: 1, 2, 4
  - `criterion`: gini, entropy

The search space is non trivial but small enough for a teaching setting.


In [ ]:
# (08) Hyperparameter tuning

# KNN hyperparameter grid
knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

knn_grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)

print("Running GridSearchCV for KNN...")
knn_grid_search.fit(X_train, y_train)
print("Best KNN params:", knn_grid_search.best_params_)
print("Best KNN CV accuracy:", knn_grid_search.best_score_)

best_knn = knn_grid_search.best_estimator_


In [ ]:
# Decision Tree hyperparameter grid
dt_param_grid = {
    "model__max_depth": [None, 3, 5, 7, 9],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"],
}

dt_grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)

print("Running GridSearchCV for Decision Tree...")
dt_grid_search.fit(X_train, y_train)
print("Best Decision Tree params:", dt_grid_search.best_params_)
print("Best Decision Tree CV accuracy:", dt_grid_search.best_score_)

best_dt = dt_grid_search.best_estimator_


# (09) Evaluation

We evaluate the tuned best estimators on the held out test set.

For each model we compute:

- Accuracy
- Precision
- Recall
- F1 score
- ROC AUC (if probabilities are available)
- Confusion matrix (values and heatmap)


In [ ]:
# (09) Evaluation

def evaluate_on_test(model, X_test, y_test, name="model"):
    y_pred = model.predict(X_test)

    metrics = {}
    metrics["accuracy"] = accuracy_score(y_test, y_pred)
    metrics["precision"] = precision_score(y_test, y_pred, zero_division=0)
    metrics["recall"] = recall_score(y_test, y_pred, zero_division=0)
    metrics["f1"] = f1_score(y_test, y_pred, zero_division=0)

    # ROC AUC if model supports probabilities
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, y_proba)
    except Exception:
        metrics["roc_auc"] = np.nan

    print(f"\n=== {name} classification report ===")
    print(classification_report(y_test, y_pred, digits=3))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0 (died)", "Actual 1 (survived)"],
        columns=["Predicted 0", "Predicted 1"],
    )
    print(f"{name} confusion matrix:")
    display(cm_df)

    plt.figure(figsize=(4, 3))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} confusion matrix")
    plt.tight_layout()
    plt.show()

    return metrics

knn_test_metrics = evaluate_on_test(best_knn, X_test, y_test, name="Tuned KNN")
dt_test_metrics = evaluate_on_test(best_dt, X_test, y_test, name="Tuned Decision Tree")

print("\nKNN test metrics:", knn_test_metrics)
print("Decision Tree test metrics:", dt_test_metrics)


Short interpretation:

- True positives are correctly identified survivors.
- True negatives are correctly identified non survivors.
- False positives are predicted as survivors but did not survive (over optimistic predictions).
- False negatives are predicted as non survivors but actually survived (more conservative predictions).

Depending on the application, one may prefer to minimize false negatives or false positives, not just maximize accuracy.


# (10) Selection of best model and conclusion

We compare the tuned KNN and tuned DecisionTree models based on test set metrics and choose a primary model. The comparison also shows trade offs between accuracy, precision, recall and ROC AUC.


In [ ]:
# (10) Selection of best model and conclusion

comparison_df = pd.DataFrame(
    {
        "Tuned KNN": knn_test_metrics,
        "Tuned Decision Tree": dt_test_metrics,
    }
)

print("Test metrics comparison:")
display(comparison_df.style.format("{:.3f}"))

# Select best model using F1 as primary metric, then accuracy as tie breaker
primary_metric = "f1"
secondary_metric = "accuracy"

primary_best = comparison_df.loc[primary_metric].idxmax()
secondary_best = comparison_df.loc[secondary_metric].idxmax()

best_model_name = primary_best
other_model_name = [m for m in comparison_df.columns if m != best_model_name][0]

print(f"\nPrimary selection metric: {primary_metric}")
print(f"Best model on test set: {best_model_name}")
print(f"Best model based on accuracy: {secondary_best}")


### Final conclusion

- The full pipeline integrates custom domain focused preprocessing (Embarked and Age imputation, cabin handling, family and ticket features, title based features) with standard scikit learn tools for scaling and encoding.
- Both KNN and DecisionTree are evaluated in the same framework, first as base models and then with tuned hyperparameters via grid search on the training set.
- The final model selection is based on test set metrics, primarily F1 score, while also considering accuracy, precision, recall and ROC AUC.
- The resulting notebook provides a compact template for industry style supervised learning on structured tabular data, not only for the Titanic dataset but for similar classification problems where clear preprocessing and modular pipelines are important.
